In [1]:
# =============================================================================
# BLOCK 1 — INPUTS (paths, tariffs, finance knobs)
# Purpose: define data locations and scenario knobs in one place.
# =============================================================================
import numpy as np
import pandas as pd
from dataclasses import dataclass, field

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 6)

# 1A) Paths to 8760 Excel files (you gave these)
PHOENIX_XLSX   = r"C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx"
FAIRBANKS_XLSX = r"C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx"

# 1B) Site tariffs (flat) — used only in annual cost calc
SITES = {
    "Phoenix":   {"energy_price_usd_per_kWh": 0.10},
    "Fairbanks": {"energy_price_usd_per_kWh": 0.08},
}

# 1C) Finance knobs (for later NPV/IRR blocks if needed)
DISCOUNT_RATE = 0.05     # 5% real
ESCALATION    = 0.02     # 2% real
LIFE_YEARS    = 20

# =============================================================================
# BLOCK 2 — DATA LOADER
# Purpose: read your Excel files and standardize columns for the engine.
# Expect input cols: ["Date/Time","Qserver_kW","Twb_in_C","Tdb_out_C"].
# For dispatch we need outdoor wet-bulb; until we have RH we use Tdb as proxy:
#   Twb_out_C ≈ Tdb_out_C   (replace later when you add RH).
# =============================================================================
def load_site_frame(xlsx_path: str) -> pd.DataFrame:
    df = pd.read_excel(xlsx_path)

    # Normalize column names
    df.columns = [c.strip().replace(" ", "") for c in df.columns]

    # Best-effort datetime (not strictly required)
    if "Date/Time" in df.columns:
        df["DateTime"] = pd.to_datetime(df["Date/Time"], errors="coerce", format="mixed")
    else:
        df["DateTime"] = pd.NaT

    out = pd.DataFrame({
        "DateTime":   df["DateTime"],
        "Qserver_kW": pd.to_numeric(df["Qserver_kW"], errors="coerce"),
        "Tdb_out_C":  pd.to_numeric(df["Tdb_out_C"],  errors="coerce"),
        "Twb_in_C":   pd.to_numeric(df.get("Twb_in_C", np.nan), errors="coerce"),
    })
    # Approximate outdoor wet-bulb with dry-bulb (temporary)
    out["Twb_out_C"] = out["Tdb_out_C"]

    # Keep exactly 8760 rows (truncate or error if fewer)
    out = out.dropna(subset=["Qserver_kW", "Tdb_out_C"]).reset_index(drop=True)
    if len(out) < 8760:
        raise ValueError(f"{xlsx_path} has only {len(out)} usable rows; need 8760.")
    if len(out) > 8760:
        out = out.iloc[:8760].copy()

    return out

df_phoenix   = load_site_frame(PHOENIX_XLSX)
df_fairbanks = load_site_frame(FAIRBANKS_XLSX)
print("Loaded source — Phoenix:", PHOENIX_XLSX)
print("Loaded source — Fairbanks:", FAIRBANKS_XLSX)
print("Loaded rows — Phoenix:", len(df_phoenix), "Fairbanks:", len(df_fairbanks))
print("Qserver first hour — Phoenix:", float(df_phoenix.loc[0, "Qserver_kW"]), "Fairbanks:", float(df_fairbanks.loc[0, "Qserver_kW"]))

# =============================================================================
# BLOCK 3 — PLANT MODEL (CRAH/WSE/CHILLER)
# Purpose: implement the equations you cite in §3.2 (hourly dispatch).
# =============================================================================
WATER_CP_KJ_PER_KG_K = 4.186
WATER_DENS_KG_PER_M3 = 1000.0

def k_cubic(x):
    """Vectorized cubic VFD scaling for fans/pumps (clamped 0..1)."""
    x = np.asarray(x, dtype=float)
    return np.clip(x, 0.0, 1.0) ** 3

def k_cubic_floor(x, power_floor=0.20):
    """Affinity with a non-zero parasitic floor if you want it; currently not used in final calc."""
    x = np.asarray(x, dtype=float)
    x = np.clip(x, 0.0, 1.0)
    return power_floor + (1.0 - power_floor) * (x**3)

def hx_eps_ntu_Q(C_hot, C_cold, UA_kW_per_K, Th_in, Tc_in):
    """
    ε–NTU HX (both-mixed approximation), vectorized.
    Inputs: C_hot, C_cold in kW/K; UA in kW/K; temperatures in °C.
    Returns: (Q_kW, Th_out_C, Tc_out_C)
    """
    C_hot  = np.asarray(C_hot,  float)
    C_cold = np.asarray(C_cold, float)
    Th_in  = np.asarray(Th_in,  float)
    Tc_in  = np.asarray(Tc_in,  float)
    UA     = float(UA_kW_per_K)
    eps    = 1e-9

    C_min = np.minimum(C_hot, C_cold)
    C_max = np.maximum(C_hot, C_cold)
    Cr    = np.divide(C_min, C_max, out=np.zeros_like(C_min), where=(C_max > eps))
    NTU   = UA / np.maximum(C_min, eps)
    term  = np.exp(-NTU * (1.0 - Cr))
    denom = 1.0 - Cr * term
    eff   = np.divide(1.0 - term, denom, out=np.zeros_like(denom), where=(denom > eps))

    Qmax   = C_min * np.maximum(0.0, Th_in - Tc_in)
    Q      = eff * Qmax
    Th_out = np.where(C_hot  > eps, Th_in - Q / np.maximum(C_hot,  eps), Th_in)
    Tc_out = np.where(C_cold > eps, Tc_in + Q / np.maximum(C_cold, eps), Tc_in)
    return Q, Th_out, Tc_out

# Chiller COP polynomial surfaces (paper Eq. 4 & 5)
def cop_centrifugal(Tctw_C, PLR):
    b1, b2, b3, b4, b5, b6 = 25.47, -1.066, 6.335, 0.01188, 0.1263, -7.581
    T = np.asarray(Tctw_C, float); L = np.asarray(PLR, float)
    return b1 + b2*T + b3*L + b4*T*T + b5*T*L + b6*L*L

# ✅ Keep only the correct magnetic COP (remove the buggy duplicate)
def cop_magnetic(Tctw_C, PLR):
    c1, c2, c3, c4, c5, c6 = 25.17, -0.7536, -1.081, 0.005736, 0.1851, -4.568
    T = np.asarray(Tctw_C, float); L = np.asarray(PLR, float)
    return c1 + c2*T + c3*L + c4*T*T + c5*T*L + c6*L*L

@dataclass
class PlantParams:
    # Water-side setpoints and tower behavior
    Tcw_supply_set_C: float = 7.0
    dT_chilled_K: float     = 5.0
    dT_cooling_K: float     = 5.0
    tower_approach_K: float = 4.0
    tower_Tmin_C: float     = 2.0

    # Water-side economizer (plate HX)
    UA_wse_kW_per_K: float = 4000.0   # UA in kW/K

    # CRAH bank (capacities & fan power)
    crah_caps_kW: tuple = (72*102.0, 29*62.0, 12*20.5)
    crah_fans_kW: tuple = (72*5.66,  29*3.44, 12*1.40)

    # Pumps (rated flows & powers)
    chw_flow_m3_per_h_per_pump: float = 660.0
    chw_pumps_use: int                = 2
    chw_pump_power_kW_per_pump: float = 75.0

    ctw_flow_m3_per_h_per_pump: float = 900.0
    ctw_pumps_use: int                = 2
    ctw_pump_power_kW_per_pump: float = 55.0

    # Tower fans (rated)
    tower_fan_power_kW_per_ct: float = 37.5
    tower_cells_use: int             = 2

    # Chillers
    chiller_type: str               = "centrifugal"   # "centrifugal" or "magnetic"
    chiller_units_use: int          = 2
    chiller_cap_kW_per_unit: float  = 4058.0          # 4058 cen; 3900 mag

    COP_min: float = 2.0
    COP_max: float = 18.0

def plan_power_timeseries(df: pd.DataFrame, params: PlantParams) -> pd.DataFrame:
    """
    Hourly dispatch (8760 rows expected):
    Inputs per row: Qserver_kW, Twb_out_C   (and Tdb_out_C for reference)
    Outputs per row: mode label, Q splits, COP, component powers, P_sys_kW.
    """
    out = df.copy()

    # Total cooling load (Eq. 2): IT + ~10% facility overhead
    Qsum = out['Qserver_kW'].to_numpy(float) * 1.10

    # CRAH fan power (affinity law)
    crah_Q_cap     = sum(params.crah_caps_kW)
    crah_fan_rated = sum(params.crah_fans_kW)
    kfan_crah      = np.clip(Qsum / max(1.0, crah_Q_cap), 0, 1)
    P_crah         = k_cubic(kfan_crah) * crah_fan_rated

    # Required chilled-water flow to satisfy Qsum at ΔT_chilled
    Cpw          = WATER_CP_KJ_PER_KG_K
    m_chw_req    = Qsum / (Cpw * params.dT_chilled_K)     # kg/s
    chw_total    = params.chw_flow_m3_per_h_per_pump * params.chw_pumps_use
    m_chw_rated  = chw_total * WATER_DENS_KG_PER_M3 / 3600.0
    kflow_chw    = np.clip(m_chw_req / max(1e-9, m_chw_rated), 0, 1)
    C_hot        = np.minimum(m_chw_req, m_chw_rated) * Cpw  # kW/K
    Tcw_return_C = params.Tcw_supply_set_C + params.dT_chilled_K

    # Tower outlet (condenser water in): Twb_out + approach, floored
    Tctw_in = np.maximum(out['Twb_out_C'].to_numpy(float) + params.tower_approach_K,
                         params.tower_Tmin_C)

    # Cooling-water side capacity (rated pumps)
    ctw_total     = params.ctw_flow_m3_per_h_per_pump * params.ctw_pumps_use
    m_ctw_rated   = ctw_total * WATER_DENS_KG_PER_M3 / 3600.0
    C_cold_rated  = m_ctw_rated * Cpw  # kW/K

    # WSE ε–NTU transfer
    Q_wse, Th_out, Tc_out = hx_eps_ntu_Q(
        C_hot=C_hot, C_cold=C_cold_rated, UA_kW_per_K=params.UA_wse_kW_per_K,
        Th_in=Tcw_return_C, Tc_in=Tctw_in
    )
    # Cap to setpoint and load
    Q_to_set = np.maximum(0.0, (Tcw_return_C - params.Tcw_supply_set_C) * C_hot)
    Q_wse    = np.minimum(np.minimum(Q_wse, Q_to_set), Qsum)

    # Mode logic + unmet load Q_need
    Q_need = np.maximum(0.0, Qsum - Q_wse)
    mode   = np.where(Q_wse >= Qsum - 1e-6, "MODE1_FULL_FREE",
             np.where(Q_wse >  1e-6,       "MODE2_PARTIAL_FREE", "MODE3_CHILLER"))

    # Chiller portion after WSE (no ATES in this block)
    Q_ch = Q_need.copy()

    # Chiller COP from condenser in temperature and PLR
    cap_unit  = params.chiller_cap_kW_per_unit
    total_cap = cap_unit * params.chiller_units_use
    PLR_total = np.clip(Q_ch / max(1e-9, total_cap), 1e-6, 1.0)

    if params.chiller_type.lower().startswith('mag'):
        COP = cop_magnetic(Tctw_in, PLR_total)
    else:
        COP = cop_centrifugal(Tctw_in, PLR_total)
    COP = np.clip(COP, params.COP_min, params.COP_max)

    P_ch = np.where(Q_ch > 0.0, Q_ch / COP, 0.0)

    # Tower rejection and auxiliaries
    Q_rej         = Q_wse + np.where(Q_ch > 0.0, Q_ch * (COP - 1.0) / COP, 0.0)
    Q_tower_rated = C_cold_rated * params.dT_cooling_K
    ktower        = np.clip(Q_rej / max(Q_tower_rated, 1e-9), 0, 1)
    kflow_ctw     = ktower

    # ✅ MOVED BELOW (after kflow_chw, kflow_ctw, ktower exist) — fixes UnboundLocalError
    P_tower_fans  = k_cubic(ktower)     * (params.tower_fan_power_kW_per_ct * params.tower_cells_use)
    P_chw_pumps   = k_cubic(kflow_chw)  * (params.chw_pump_power_kW_per_pump * params.chw_pumps_use)
    P_ctw_pumps   = k_cubic(kflow_ctw)  * (params.ctw_pump_power_kW_per_pump * params.ctw_pumps_use)

    P_sys = P_ch + P_chw_pumps + P_ctw_pumps + P_tower_fans + P_crah

    out = out.assign(
        Qsum_kW=Qsum,
        Q_need_kW=Q_need,
        mode=mode,
        Q_wse_kW=Q_wse,
        Q_chiller_kW=Q_ch,
        COP=np.where(Q_ch > 0.0, COP, np.nan),
        P_chiller_kW=P_ch,
        P_crah_kW=P_crah,
        P_pumps_chw_kW=P_chw_pumps,
        P_pumps_ctw_kW=P_ctw_pumps,
        P_tower_fans_kW=P_tower_fans,
        P_sys_kW=P_sys,
        T_ctw_in_C=Tctw_in,
        T_cw_return_C=Tcw_return_C,
        T_cw_after_WSE_C=(Tcw_return_C - np.where(C_hot>0, Q_wse/np.maximum(C_hot,1e-9), 0.0))
    )
    return out

# =============================================================================
# BLOCK 4 — EXEC KPIs + PLAN COMPARISON
# Purpose: compute side-by-side hourly power and annual kWh/$ for Cen vs Mag.
# =============================================================================
def add_exec_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['Qserver_kW']       = out['Qsum_kW'] / 1.10
    out['P_chiller_kW_est'] = np.where(out['Q_chiller_kW']>0, out['Q_chiller_kW']/out['COP'], 0.0)
    out['P_aux_kW']         = out['P_sys_kW'] - out['P_chiller_kW_est']
    out['Cooling_overhead_%'] = 100.0 * out['P_sys_kW'] / out['Qserver_kW']
    out['PUE_prime']        = 1.0 + out['P_sys_kW'] / out['Qserver_kW']
    return out

def compare_plans(df_hourly, p_mag, p_cen, elec_price: float, dt_hours: float = 1.0):
    res_mag = plan_power_timeseries(df_hourly, p_mag)
    res_cen = plan_power_timeseries(df_hourly, p_cen)

    view = pd.DataFrame({
        'mode': res_mag['mode'],
        'IT_kW': res_mag['Qsum_kW'] / 1.10,
        'Twb_out_C': df_hourly['Twb_out_C'],
        'P_sys_mag_kW': res_mag['P_sys_kW'],
        'P_sys_cen_kW': res_cen['P_sys_kW'],
    })
    view['Δ_kW (cen - mag)'] = view['P_sys_cen_kW'] - view['P_sys_mag_kW']

    kWh_mag   = float((res_mag['P_sys_kW'] * dt_hours).sum())
    kWh_cen   = float((res_cen['P_sys_kW'] * dt_hours).sum())
    delta_kWh = kWh_cen - kWh_mag
    delta_usd = delta_kWh * float(elec_price)
    pct_sav   = 100.0 * delta_kWh / kWh_cen if kWh_cen > 0 else np.nan

    totals = {
        "Magnetic total kWh":     kWh_mag,
        "Centrifugal total kWh":  kWh_cen,
        "ΔkWh (cen - mag)":       delta_kWh,
        "ΔUSD (cen - mag)":       delta_usd,
        "% savings vs centrifugal": pct_sav
    }
    return res_mag, res_cen, view, totals

# =============================================================================
# BLOCK 5 — RUN (8760 outputs) AND PRINT ANNUAL KPIs
# Purpose: produce exactly 8760-row hourly tables + compact annual summaries.
# =============================================================================
# Plant parameter bundles (only difference is chiller type/capacity)
p_mag = PlantParams(chiller_type="magnetic",    chiller_cap_kW_per_unit=3900.0)
p_cen = PlantParams(chiller_type="centrifugal", chiller_cap_kW_per_unit=4058.0)

# --- Phoenix
res_mag_phx, res_cen_phx, view_phx, totals_phx = compare_plans(
    df_phoenix, p_mag, p_cen, elec_price=SITES["Phoenix"]["energy_price_usd_per_kWh"], dt_hours=1.0
)
print("\n=== PHOENIX — Hourly (first 6 of 8760) ===")
print(view_phx.head(6))
print("\n=== PHOENIX — Annual totals ===")
for k, v in totals_phx.items():
    print(f"{k}: {v:,.2f}")

# --- Fairbanks
res_mag_fb, res_cen_fb, view_fb, totals_fb = compare_plans(
    df_fairbanks, p_mag, p_cen, elec_price=SITES["Fairbanks"]["energy_price_usd_per_kWh"], dt_hours=1.0
)
print("\n=== FAIRBANKS — Hourly (first 6 of 8760) ===")
print(view_fb.head(6))
print("\n=== FAIRBANKS — Annual totals ===")
for k, v in totals_fb.items():
    print(f"{k}: {v:,.2f}")


Loaded source — Phoenix: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx
Loaded source — Fairbanks: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx
Loaded rows — Phoenix: 8760 Fairbanks: 8760
Qserver first hour — Phoenix: 1342.595476084541 Fairbanks: 1342.595476084541

=== PHOENIX — Hourly (first 6 of 8760) ===
                 mode        IT_kW  Twb_out_C  P_sys_mag_kW  P_sys_cen_kW  Δ_kW (cen - mag)
0       MODE3_CHILLER  1342.595476      11.18    100.594686    114.760547         14.165861
1       MODE3_CHILLER  1342.685053       9.79     95.943290    107.190236         11.246946
2       MODE3_CHILLER  1340.395705       8.49     91.698479    100.542454          8.843975
3  MODE2_PARTIAL_FREE  1338.474414       7.79     85.843891     93.438577          7.594686
4       MODE3_CHILLER  1339.993680       8.03     90.290431     98.355683          8.065253
5  MODE2_PARTIAL_FREE  1335.405900       6.58  

In [2]:
# =========================
# CLEAN, ROBUST PRINTER FOR tbl_fin (no math changes) — trimmed columns
# =========================
import numpy as np
import pandas as pd
from typing import Optional, List

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

def _fmt_money(x):
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return "—"
        return f"${x:,.0f}"
    except Exception:
        return str(x)

def _fmt_kwh(x):
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return "—"
        return f"{x:,.0f}"
    except Exception:
        return str(x)

def _fmt_tariff(x):
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return "—"
        return f"${x:.3f}/kWh"
    except Exception:
        return str(x)

def _fmt_years(x):
    try:
        if x is None or (isinstance(x, float) and not np.isfinite(x)):
            return "—"
        return f"{x:,.1f}"
    except Exception:
        return str(x)

def _find_col(df: pd.DataFrame, must_have: List[str], any_of: Optional[List[str]] = None) -> Optional[str]:
    cols = list(df.columns)
    cols_lc = [str(c).lower() for c in cols]
    for c, lc in zip(cols, cols_lc):
        if all(s in lc for s in must_have) and (any_of is None or any(s in lc for s in any_of)):
            return c
    return None

def print_finance_view(tbl_fin: pd.DataFrame) -> pd.DataFrame:
    """Pretty-print incremental finance view for Magnetic vs Centrifugal.
       Returns the pretty DataFrame for further use."""
    if not isinstance(tbl_fin, pd.DataFrame):
        raise ValueError("print_finance_view: expected a pandas DataFrame")

    # Auto-detect columns by keywords (case/spacing tolerant)
    col_site        = _find_col(tbl_fin, ["site"])
    col_kwh_cen     = _find_col(tbl_fin, ["kwh"], any_of=["cen","centrifugal"])
    col_kwh_mag     = _find_col(tbl_fin, ["kwh"], any_of=["mag","magnetic"])
    # col_dkwh intentionally NOT used (you asked to omit ΔkWh)
    col_tariff      = _find_col(tbl_fin, ["tariff"]) or _find_col(tbl_fin, ["/kwh"])
    col_energy_sav  = _find_col(tbl_fin, ["energy","saving"])
    # col_premium_cap intentionally NOT used (omit Premium CAPEX)
    # col_ann_prem   intentionally NOT used (omit Annualized premium/yr)
    col_delta_om    = _find_col(tbl_fin, ["o&m"], any_of=["Δ","delta","premium"])
    col_net_sav     = _find_col(tbl_fin, ["net"], any_of=["/yr","annual","year"])
    col_npv         = _find_col(tbl_fin, ["npv"])
    col_spb         = _find_col(tbl_fin, ["payback"])

    # Build pretty view only with the columns you want (skip missing gracefully)
    pretty_cols = {}
    if col_site:        pretty_cols["Site"]                 = tbl_fin[col_site]
    if col_kwh_cen:     pretty_cols["kWh (Centrifugal)"]    = tbl_fin[col_kwh_cen].apply(_fmt_kwh)
    if col_kwh_mag:     pretty_cols["kWh (Magnetic)"]       = tbl_fin[col_kwh_mag].apply(_fmt_kwh)
    if col_tariff:      pretty_cols["Tariff"]               = tbl_fin[col_tariff].apply(_fmt_tariff)
    if col_energy_sav:  pretty_cols["Energy savings / yr"]  = tbl_fin[col_energy_sav].apply(_fmt_money)
    if col_delta_om:    pretty_cols["ΔO&M / yr"]            = tbl_fin[col_delta_om].apply(_fmt_money)
    if col_net_sav:     pretty_cols["Net savings / yr"]     = tbl_fin[col_net_sav].apply(_fmt_money)
    if col_npv:         pretty_cols["NPV (r, N)"]           = tbl_fin[col_npv].apply(_fmt_money)
    if col_spb:         pretty_cols["Simple payback (yrs)"] = tbl_fin[col_spb].apply(_fmt_years)

    view_pretty = pd.DataFrame(pretty_cols)

    print("\n=== Incremental Finance: Magnetic vs Centrifugal — Clean View ===")
    if view_pretty.empty:
        print("(No matching columns found — check your headers.)")
    else:
        print(view_pretty.to_string(index=False))

    # Friendly note for any infinite/undefined paybacks
    if col_spb and col_net_sav:
        for i, row in tbl_fin.iterrows():
            spb_val = row[col_spb]
            net_val = row[col_net_sav] if isinstance(row[col_net_sav], (int, float, np.floating)) else np.nan
            if not (isinstance(spb_val, (int, float, np.floating)) and np.isfinite(spb_val)):
                site_val = row[col_site] if col_site else f"Row {i}"
                note = (f"\nNote for {site_val}: Simple payback is '—' because annual net savings "
                        f"({net_val:+,.0f} $/yr) are ≤ 0 with current tariff/premium.")
                print(note)

    return view_pretty

# =========================
# DEFINE tbl_fin FROM FRESH MODEL OUTPUTS (Phoenix + Fairbanks, Mag vs Cen)
# =========================

# Keep this table tied to the current run. Do not paste old kWh/finance values here.
CHILLER_CAPEX_PER_TON_CEN_USD = 800.0
CHILLER_CAPEX_PER_TON_MAG_USD = CHILLER_CAPEX_PER_TON_CEN_USD * 1.311
BASELINE_OPEX_PCT_PER_YEAR = 0.05

def npv_of_savings_usd(annual_savings_usd: float, years=LIFE_YEARS, d=DISCOUNT_RATE, esc=ESCALATION) -> float:
    annual_savings_usd = max(0.0, float(annual_savings_usd))
    return sum(annual_savings_usd * ((1 + esc) ** (t - 1)) / ((1 + d) ** t) for t in range(1, int(years) + 1))

def chiller_capex_usd_from_peak_Q(peak_Q_kW: float, per_ton_usd: float) -> float:
    return (max(0.0, float(peak_Q_kW)) / 3.517) * max(0.0, float(per_ton_usd))

def _finance_row_from_current_results(site: str, res_mag: pd.DataFrame, res_cen: pd.DataFrame, totals: dict) -> dict:
    tariff = float(SITES[site]["energy_price_usd_per_kWh"])
    kwh_cen = float(totals["Centrifugal total kWh"])
    kwh_mag = float(totals["Magnetic total kWh"])
    energy_savings = float(totals["ΔUSD (cen - mag)"])

    # Size premium from the current peak cooling load, not from old printed tables.
    qsize_kW = float(pd.concat([res_cen["Qsum_kW"], res_mag["Qsum_kW"]]).max())
    capex_cen = chiller_capex_usd_from_peak_Q(qsize_kW, CHILLER_CAPEX_PER_TON_CEN_USD)
    capex_mag = chiller_capex_usd_from_peak_Q(qsize_kW, CHILLER_CAPEX_PER_TON_MAG_USD)
    premium_capex = capex_mag - capex_cen
    delta_om = BASELINE_OPEX_PCT_PER_YEAR * premium_capex

    return {
        "Site": site,
        "kWh (Centrifugal)": kwh_cen,
        "kWh (Magnetic)": kwh_mag,
        "Tariff $/kWh": tariff,
        "Energy savings / yr": energy_savings,
        "Premium CapEx $": premium_capex,
        "ΔO&M / yr": delta_om,
        "Net savings / yr": energy_savings - delta_om,
        "NPV (r, N)": npv_of_savings_usd(energy_savings),
        "Simple payback (yrs)": (premium_capex / energy_savings) if energy_savings > 0 else float("inf"),
    }

tbl_fin = pd.DataFrame([
    _finance_row_from_current_results("Phoenix", res_mag_phx, res_cen_phx, totals_phx),
    _finance_row_from_current_results("Fairbanks", res_mag_fb, res_cen_fb, totals_fb),
])

# ---- Example usage (now it will work, because tbl_fin exists)
view = print_finance_view(tbl_fin)



=== Incremental Finance: Magnetic vs Centrifugal — Clean View ===
     Site kWh (Centrifugal) kWh (Magnetic)     Tariff Energy savings / yr NPV (r, N) Simple payback (yrs)
  Phoenix         2,294,010      1,706,964 $0.100/kWh             $58,705   $860,926                  2.1
Fairbanks           520,452        429,382 $0.080/kWh              $7,286   $106,846                 17.0


In [3]:
# # =========================
# # CLEAN, ROBUST PRINTER FOR tbl_fin (no math changes) — trimmed columns
# # =========================
# import numpy as np
# import pandas as pd
# from typing import Optional, List

# pd.set_option("display.width", 160)
# pd.set_option("display.max_columns", 60)

# def _fmt_money(x):
#     try:
#         if x is None or (isinstance(x, float) and not np.isfinite(x)):
#             return "—"
#         return f"${x:,.0f}"
#     except Exception:
#         return str(x)

# def _fmt_kwh(x):
#     try:
#         if x is None or (isinstance(x, float) and not np.isfinite(x)):
#             return "—"
#         return f"{x:,.0f}"
#     except Exception:
#         return str(x)

# def _fmt_tariff(x):
#     try:
#         if x is None or (isinstance(x, float) and not np.isfinite(x)):
#             return "—"
#         return f"${x:.3f}/kWh"
#     except Exception:
#         return str(x)

# def _fmt_years(x):
#     try:
#         if x is None or (isinstance(x, float) and not np.isfinite(x)):
#             return "—"
#         return f"{x:,.1f}"
#     except Exception:
#         return str(x)

# def _find_col(df: pd.DataFrame, must_have: List[str], any_of: Optional[List[str]] = None) -> Optional[str]:
#     cols = list(df.columns)
#     cols_lc = [str(c).lower() for c in cols]
#     for c, lc in zip(cols, cols_lc):
#         if all(s in lc for s in must_have) and (any_of is None or any(s in lc for s in any_of)):
#             return c
#     return None

# def print_finance_view(tbl_fin: pd.DataFrame) -> pd.DataFrame:
#     """Pretty-print incremental finance view for Magnetic vs Centrifugal.
#        Returns the pretty DataFrame for further use."""
#     if not isinstance(tbl_fin, pd.DataFrame):
#         raise ValueError("print_finance_view: expected a pandas DataFrame")

#     # Auto-detect columns by keywords (case/spacing tolerant)
#     col_site        = _find_col(tbl_fin, ["site"])
#     col_kwh_cen     = _find_col(tbl_fin, ["kwh"], any_of=["cen","centrifugal"])
#     col_kwh_mag     = _find_col(tbl_fin, ["kwh"], any_of=["mag","magnetic"])
#     # col_dkwh intentionally NOT used (you asked to omit ΔkWh)
#     col_tariff      = _find_col(tbl_fin, ["tariff"]) or _find_col(tbl_fin, ["/kwh"])
#     col_energy_sav  = _find_col(tbl_fin, ["energy","saving"])
#     # col_premium_cap intentionally NOT used (omit Premium CAPEX)
#     # col_ann_prem   intentionally NOT used (omit Annualized premium/yr)
#     col_delta_om    = _find_col(tbl_fin, ["o&m"], any_of=["Δ","delta","premium"])
#     col_net_sav     = _find_col(tbl_fin, ["net"], any_of=["/yr","annual","year"])
#     col_npv         = _find_col(tbl_fin, ["npv"])
#     col_spb         = _find_col(tbl_fin, ["payback"])

#     # Build pretty view only with the columns you want (skip missing gracefully)
#     pretty_cols = {}
#     if col_site:        pretty_cols["Site"]                 = tbl_fin[col_site]
#     if col_kwh_cen:     pretty_cols["kWh (Centrifugal)"]    = tbl_fin[col_kwh_cen].apply(_fmt_kwh)
#     if col_kwh_mag:     pretty_cols["kWh (Magnetic)"]       = tbl_fin[col_kwh_mag].apply(_fmt_kwh)
#     if col_tariff:      pretty_cols["Tariff"]               = tbl_fin[col_tariff].apply(_fmt_tariff)
#     if col_energy_sav:  pretty_cols["Energy savings / yr"]  = tbl_fin[col_energy_sav].apply(_fmt_money)
#     if col_delta_om:    pretty_cols["ΔO&M / yr"]            = tbl_fin[col_delta_om].apply(_fmt_money)
#     if col_net_sav:     pretty_cols["Net savings / yr"]     = tbl_fin[col_net_sav].apply(_fmt_money)
#     if col_npv:         pretty_cols["NPV (r, N)"]           = tbl_fin[col_npv].apply(_fmt_money)
#     if col_spb:         pretty_cols["Simple payback (yrs)"] = tbl_fin[col_spb].apply(_fmt_years)

#     view_pretty = pd.DataFrame(pretty_cols)

#     print("\n=== Incremental Finance: Magnetic vs Centrifugal — Clean View ===")
#     if view_pretty.empty:
#         print("(No matching columns found — check your headers.)")
#     else:
#         print(view_pretty.to_string(index=False))

#     # Friendly note for any infinite/undefined paybacks
#     if col_spb and col_net_sav:
#         for i, row in tbl_fin.iterrows():
#             spb_val = row[col_spb]
#             net_val = row[col_net_sav] if isinstance(row[col_net_sav], (int, float, np.floating)) else np.nan
#             if not (isinstance(spb_val, (int, float, np.floating)) and np.isfinite(spb_val)):
#                 site_val = row[col_site] if col_site else f"Row {i}"
#                 note = (f"\nNote for {site_val}: Simple payback is '—' because annual net savings "
#                         f"({net_val:+,.0f} $/yr) are ≤ 0 with current tariff/premium.")
#                 print(note)

#     return view_pretty

# # ---- Example usage:
# #view = print_finance_view(tbl_fin)  # pass your DataFrame variable here
